## Exploring architectural design choices

The structure should be this:
- I should have a file `models/` and each of this implements a single model. Starts with gpt and ends with recursion, slowly adding/removing one component up until a full recursion process is done.
- I should have a training task where I select which model is trained on and what dataset is trained on
- The training of the model and dataset should be independent arguments and I should be able to just select which one I am doing: which model on which dataset.
- The evaluation should also be clear, with the ability to evaluate both the losses on the val dataset as well as performance (e.g. if it's a math task) by just selecting the best model
- The saving of the files should also be clear, i.e. the best model should be saved.

What we need to fix:
- Same dataset
- Same prediction task
- Same loss
- Match parameter counts

So setup where:
- Different models
- Different datasets

---
What are the different models I have?
1. Model 1: Vanilla GPT
2. Model 2: Two-stream GPT
    - What is the effect of having a two-stream architecture, even without recursion? 
    - We add reasoning <- f_theta (reasoning, solution + input_seq)
    - We add solution <- f_theata (solution, reasoning)
    - GPT mixes everything into a single stream. M2 forces information to bounce between reasoning and solution.
3. Model 3: Reuse the same UpdateNEtwork multiple times per layer but keep gradients through all cycles to see pure recurrsence effects. Does “more depth via recurrence of the same block” help compared to a feedforward stack at fixed compute?
4. Model 4: Many recurrences cycles and one update
5. Model 5: Many currences cycles, one update, repeated multiple times
6. Model 6: Adding halting logic
---
Model 1: Vanilla GPT
- Key change: Standard single-stream Transformer decoder with causal self-attention and a feedforward stack; no recursion or auxiliary state.
- Purpose: Establish a strong, well-understood baseline for next-token prediction against which all recursive variants can be compared in terms of accuracy and compute.

Model 2: Two-Stream GPT (Solution–Reasoning Split)
- Key change: Replace the single hidden state with two coupled streams, updating reasoning ← fθ(reasoning, solution + input_seq) and solution ← fθ(solution, reasoning) once per layer, without any extra recurrence.
- Purpose: Isolate the effect of explicitly separating “working memory” (reasoning) from “answer representation” (solution) while holding total depth and compute similar to vanilla GPT.

Model 3: GPT with Recurrent Depth (Shared Update Network)
- Key change: Reuse the same UpdateNetwork multiple times per layer (inner cycles over reasoning and solution) with gradients flowing through all cycles, effectively increasing depth without adding new parameters.
- Purpose: Test whether greater effective depth via recurrence of a shared block improves modeling capacity compared to a purely feedforward stack at similar parameter count and compute.

Model 4: TR-GPT Warm-Up Recurrence (Many Cycles, One Gradient-Carrying Update)
- Key change: Introduce TRM-style recurrence where several outer cycles run under no_grad (warm-up) and only the final cycle carries gradients, with each cycle performing multiple reasoning updates followed by a solution update.
- Purpose: Probe whether cheap, gradient-free iterative refinement toward a fixed point before a single learned update helps or hurts language modeling performance.

Model 5:TR-GPT Multi-Step Supervision with Fixed Recurrence (Many Cycles, Repeated Updates)
- Key change: Keep the TRM warm-up recurrence per step, but apply it multiple times in an outer supervision loop, reusing the latent (solution, reasoning) state and optionally applying deep supervision across steps.
- Purpose: Study whether training the model to iteratively refine its solution over several supervised steps (with persistent latent state) yields better representations than a single-step prediction.

Model 6: TR-GPT with ACT-Style Halting (Adaptive Computation)
- Key change: Add a halting head on the solution stream and ACT-style logic that decides, per example, how many outer supervision steps to run up to a maximum, optionally regularized toward shorter computation.
- Purpose: Examine whether adaptive compute—allocating more recursion to difficult inputs and less to easy ones—improves the accuracy–compute trade-off compared to fixed-depth recursive models.


In [10]:
from dataclasses import dataclass

@dataclass
class CharTokenizer:
    stoi: dict
    itos: dict

    @property
    def vocab_size(self) -> int:
        return len(self.stoi)

    def encode(self, s: str):
        return [self.stoi[c] for c in s]

    def decode(self, ids):
        return "".join([self.itos[int(i)] for i in ids])

In [13]:

import sys
import os
# Add parent directory to path so we can import from callbacks and data_modules
sys.path.insert(0, os.path.abspath('..'))

# Data visualization on algorithmic reasoning

In [2]:
import torch
import matplotlib.pyplot as plt

# from datasets.copy_char import load_copy_char
from data_modules.algorithmic_char import load_algorithmic_char


/home/azureuser/miniconda/envs/trm2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
BATCH_SIZE = 4
BLOCK_SIZE = 32
SEED = 42

train_loader, val_loader, tokenizer = load_algorithmic_char(
    task="reverse", # reverse, addition, copy
    data_dir="data",
    block_size=BLOCK_SIZE,
    batch_size=BATCH_SIZE,
    eval_iters=100,
    seed=SEED,
    train_seq_len=3
)

In [10]:
# show some examples
X, Y = next(iter(train_loader))

# Show shapes
print(X.shape)
print(Y.shape)



torch.Size([4, 32])
torch.Size([4, 32])


In [11]:
print("---- X ----")
print(tokenizer.decode(X[0]))
print("---- Y----")
print(tokenizer.decode(Y[0]))

# Show tokenizer vocab size
#print(tokenizer.vocab_size)



---- X ----
|558
083|380
577|775
329|923
699
---- Y----
558
083|380
577|775
329|923
699|


In [8]:
# Compare X and Y for the first few characters of the first sample
sample_idx = 0
x_tokens = X[sample_idx].tolist()
y_tokens = Y[sample_idx].tolist()

print("\n--- Next-Token Prediction Verification ---")
for t in range(5): # Check first 5 positions
    curr_char = tokenizer.decode([x_tokens[t]])
    target_char = tokenizer.decode([y_tokens[t]])
    print(f"Input: '{curr_char}' -> Target: '{target_char}'")


--- Next-Token Prediction Verification ---
Input: '=' -> Target: '0'
Input: '0' -> Target: '5'
Input: '5' -> Target: '6'
Input: '6' -> Target: '0'
Input: '0' -> Target: '
'


In [ ]:
# Exploring Algorithmic Tasks: Copy, Reverse, and Addition
# =========================================================

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
from data_modules.algorithmic_char import load_algorithmic_char, _build_tokenizer, _make_example

BATCH_SIZE = 4
BLOCK_SIZE = 32
SEED = 42
SEQ_LEN = 3  # Length of digits in examples

print("=" * 80)
print("EXPLORING ALGORITHMIC TASKS")
print("=" * 80)

# Load data for each task
tasks = ["copy", "reverse", "addition"]

for task in tasks:
    print(f"\n{'='*80}")
    print(f"TASK: {task.upper()}")
    print(f"{'='*80}")
    
    # Load data
    train_loader, val_loader, tokenizer = load_algorithmic_char(
        task=task,
        data_dir="data",
        block_size=BLOCK_SIZE,
        batch_size=BATCH_SIZE,
        eval_iters=10,
        seed=SEED,
        train_seq_len=SEQ_LEN,
        val_seq_len=SEQ_LEN,
    )
    
    # Get a batch
    X, Y = next(iter(train_loader))
    
    print(f"\nBatch shapes: X={X.shape}, Y={Y.shape}")
    print(f"\nSample examples from the batch:")
    print("-" * 80)
    
    # Show first 3 examples
    for i in range(min(3, X.shape[0])):
        x_str = tokenizer.decode(X[i])
        y_str = tokenizer.decode(Y[i])
        
        print(f"\nExample {i+1}:")
        print(f"  X (input):  '{x_str}'")
        print(f"  Y (target): '{y_str}'")
        
        # Show the relationship
        if task == "copy":
            # Extract the pattern: input|output
            if "|" in x_str:
                parts = x_str.split("|")
                if len(parts) >= 1:
                    input_part = parts[0]
                    # Find where the output starts in Y
                    if "|" in y_str:
                        output_part = y_str.split("|")[1] if len(y_str.split("|")) > 1 else y_str
                    else:
                        output_part = y_str
                    print(f"  → Copy task: input '{input_part}' should be copied to '{output_part}'")
                    print(f"    ✓ Correct" if input_part == output_part else f"    ✗ Incorrect")
        
        elif task == "reverse":
            if "|" in x_str:
                parts = x_str.split("|")
                if len(parts) >= 1:
                    input_part = parts[0]
                    if "|" in y_str:
                        output_part = y_str.split("|")[1] if len(y_str.split("|")) > 1 else y_str
                    else:
                        output_part = y_str
                    expected = input_part[::-1]
                    print(f"  → Reverse task: input '{input_part}' should be reversed to '{expected}'")
                    print(f"    Got: '{output_part}'")
                    print(f"    ✓ Correct" if expected == output_part else f"    ✗ Incorrect")
        
        elif task == "addition":
            if "+" in x_str and "=" in x_str:
                # Extract a+b= from X, and c from Y
                parts = x_str.split("+")
                if len(parts) == 2:
                    a = parts[0]
                    b_and_eq = parts[1]
                    if "=" in b_and_eq:
                        b = b_and_eq.split("=")[0]
                        # Find the result in Y
                        if "=" in y_str:
                            result = y_str.split("=")[1] if len(y_str.split("=")) > 1 else y_str
                        else:
                            result = y_str
                        expected = int(a) + int(b)
                        expected_str = f"{expected:0{len(a)+1}d}"
                        print(f"  → Addition task: {a} + {b} = {expected}")
                        print(f"    Expected: '{expected_str}', Got: '{result}'")
                        print(f"    ✓ Correct" if expected_str == result else f"    ✗ Incorrect")
≠

EXPLORING ALGORITHMIC TASKS

TASK: COPY

Batch shapes: X=torch.Size([4, 32]), Y=torch.Size([4, 32])

Sample examples from the batch:
--------------------------------------------------------------------------------

Example 1:
  X (input):  '|855
083|083
577|577
329|329
699'
  Y (target): '855
083|083
577|577
329|329
699|'
  → Copy task: input '' should be copied to '083
577'
    ✗ Incorrect

Example 2:
  X (input):  '637
107|107
752|752
135|135
078|'
  Y (target): '37
107|107
752|752
135|135
078|0'
  → Copy task: input '637
107' should be copied to '107
752'
    ✗ Incorrect

Example 3:
  X (input):  '56
835|835
583|583
895|895
553|5'
  Y (target): '6
835|835
583|583
895|895
553|55'
  → Copy task: input '56
835' should be copied to '835
583'
    ✗ Incorrect

TASK: REVERSE

Batch shapes: X=torch.Size([4, 32]), Y=torch.Size([4, 32])

Sample examples from the batch:
--------------------------------------------------------------------------------

Example 1:
  X (input):  '|558
083|380
577|

In [ ]:
# Exploring Algorithmic Tasks: Copy, Reverse, and Addition
# =========================================================

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
from data_modules.algorithmic_char import load_algorithmic_char, _build_tokenizer, _make_example

BATCH_SIZE = 4
BLOCK_SIZE = 32
SEED = 42
SEQ_LEN = 3  # Length of digits in examples

print("=" * 80)
print("EXPLORING ALGORITHMIC TASKS")
print("=" * 80)

# Load data for each task
tasks = ["copy", "reverse", "addition"]

for task in tasks:
    print(f"\n{'='*80}")
    print(f"TASK: {task.upper()}")
    print(f"{'='*80}")
    
    # Load data
    train_loader, val_loader, tokenizer = load_algorithmic_char(
        task=task,
        data_dir="data",
        block_size=BLOCK_SIZE,
        batch_size=BATCH_SIZE,
        eval_iters=10,
        seed=SEED,
        train_seq_len=SEQ_LEN,
        val_seq_len=SEQ_LEN,
    )
    
    # Get a batch
    X, Y = next(iter(train_loader))
    
    print(f"\nBatch shapes: X={X.shape}, Y={Y.shape}")
    print(f"\nSample examples from the batch:")
    print("-" * 80)
    
    # Show first 3 examples
    for i in range(min(3, X.shape[0])):
        x_str = tokenizer.decode(X[i])
        y_str = tokenizer.decode(Y[i])
        
        print(f"\nExample {i+1}:")
        print(f"  X (input):  '{x_str}'")
        print(f"  Y (target): '{y_str}'")
        
        # Show the relationship
        if task == "copy":
            # Extract the pattern: input|output
            if "|" in x_str:
                parts = x_str.split("|")
                if len(parts) >= 1:
                    input_part = parts[0]
                    # Find where the output starts in Y
                    if "|" in y_str:
                        output_part = y_str.split("|")[1] if len(y_str.split("|")) > 1 else y_str
                    else:
                        output_part = y_str
                    print(f"  → Copy task: input '{input_part}' should be copied to '{output_part}'")
                    print(f"    ✓ Correct" if input_part == output_part else f"    ✗ Incorrect")
        
        elif task == "reverse":
            if "|" in x_str:
                parts = x_str.split("|")
                if len(parts) >= 1:
                    input_part = parts[0]
                    if "|" in y_str:
                        output_part = y_str.split("|")[1] if len(y_str.split("|")) > 1 else y_str
                    else:
                        output_part = y_str
                    expected = input_part[::-1]
                    print(f"  → Reverse task: input '{input_part}' should be reversed to '{expected}'")
                    print(f"    Got: '{output_part}'")
                    print(f"    ✓ Correct" if expected == output_part else f"    ✗ Incorrect")
        
        elif task == "addition":
            if "+" in x_str and "=" in x_str:
                # Extract a+b= from X, and c from Y
                parts = x_str.split("+")
                if len(parts) == 2:
                    a = parts[0]
                    b_and_eq = parts[1]
                    if "=" in b_and_eq:
                        b = b_and_eq.split("=")[0]
                        # Find the result in Y
                        if "=" in y_str:
                            result = y_str.split("=")[1] if len(y_str.split("=")) > 1 else y_str
                        else:
                            result = y_str
                        expected = int(a) + int(b)
                        expected_str = f"{expected:0{len(a)+1}d}"
                        print(f"  → Addition task: {a} + {b} = {expected}")
                        print(f"    Expected: '{expected_str}', Got: '{result}'")
                        print(f"    ✓ Correct" if expected_str == result else f"    ✗ Incorrect")

# Manual inspection: Generate specific examples
print(f"\n{'='*80}")
print("MANUAL INSPECTION: Generating specific examples")
print(f"{'='*80}")

gen = torch.Generator().manual_seed(42)
tok = _build_tokenizer()

for task in tasks:
    print(f"\n{'-'*80}")
    print(f"Task: {task.upper()}")
    print(f"{'-'*80}")
    
    # Generate 5 examples
    for i in range(5):
        prompt, target, full = _make_example(task, L=SEQ_LEN, gen=gen)
        print(f"\nExample {i+1}:")
        print(f"  Prompt: '{prompt}'")
        print(f"  Target: '{target}'")
        print(f"  Full:   '{full}'")
        
        # Show what correct vs incorrect would look like
        if task == "copy":
            print(f"  ✓ Correct prediction: '{target}' (same as input)")
            print(f"  ✗ Wrong prediction:   '{target[::-1]}' (reversed)")
        elif task == "reverse":
            input_part = prompt.split("|")[0]
            print(f"  ✓ Correct prediction: '{target}' (reversed from '{input_part}')")
            print(f"  ✗ Wrong prediction:   '{input_part}' (not reversed)")
        elif task == "addition":
            parts = prompt.split("+")
            a = parts[0]
            b = parts[1].split("=")[0]
            wrong_result = int(a) + int(b) + 1  # Off by one
            wrong_str = f"{wrong_result:0{len(a)+1}d}"
            print(f"  ✓ Correct prediction: '{target}' ({a} + {b} = {int(a) + int(b)})")
            print(f"  ✗ Wrong prediction:   '{wrong_str}' (off by one)")

# Understanding the evaluation format
print(f"\n{'='*80}")
print("UNDERSTANDING EVALUATION FORMAT")
print(f"{'='*80}")

print("""
During evaluation:
1. Model receives PROMPT (e.g., "123|" for copy/reverse, or "12+34=" for addition)
2. Model generates OUTPUT tokens (max_new_tokens = L for copy/reverse, L+1 for addition)
3. Evaluation compares:
   - Character-level accuracy: how many characters match
   - Sequence-level accuracy: does the entire output match exactly?

For copy task (L=3):
  Prompt: "123|"
  Target: "123"
  ✓ Correct: "123" → 100% char acc, 100% seq acc
  ✗ Wrong:   "321" → 0% char acc, 0% seq acc
  ✗ Partial: "12X" → 66.7% char acc, 0% seq acc

For reverse task (L=3):
  Prompt: "123|"
  Target: "321"
  ✓ Correct: "321" → 100% char acc, 100% seq acc
  ✗ Wrong:   "123" → 0% char acc, 0% seq acc
  ✗ Partial: "32X" → 66.7% char acc, 0% seq acc

For addition task (L=2):
  Prompt: "12+34="
  Target: "046" (12 + 34 = 46, padded to 3 digits)
  ✓ Correct: "046" → 100% char acc, 100% seq acc
  ✗ Wrong:   "047" → 66.7% char acc, 0% seq acc (off by one)
  ✗ Wrong:   "0460" → length mismatch, marked as incorrect
""")

print("\n" + "=" * 80)
print("You can now manually test predictions by comparing strings!")
print("=" * 80)

In [ ]:
# Understanding the evaluation
